# Recherche documentaire sur SciFact

L'idée est de retrouver, pour une affirmation scientifique, les articles qui
permettent de la vérifier. SciFact fait partie du banc BEIR, donc il y a des
chiffres publiés auxquels se comparer.

Avant d'essayer d'améliorer quoi que ce soit, je reproduis trois systèmes connus.
Si mes chiffres tombent à côté des leurs, c'est ma chaîne de mesure qui est
fausse et tout ce qui suit ne vaut rien.

| | nDCG@10 publié |
|---|---|
| BM25 | 0,665 |
| BM25 + cross-encodeur | 0,688 |
| E5-large-v2 | 0,7224 |
| BGE-large-en-v1.5 | 0,7461 |
| GTE-large-en-v1.5 | 0,8243 |

Avant de lancer : Exécution, Modifier le type d'exécution, GPU.

GTE charge du code maison écrit pour transformers 4. Sur transformers 5 il
plante avec une erreur CUDA qui empoisonne tout le processus. On épingle la
version, et il faut redémarrer la session après cette cellule.

In [ ]:
!pip -q install "transformers>=4.44,<5" "sentence-transformers>=3.0,<4" nltk 2>&1 | tail -2
print("Redémarrer la session maintenant, puis reprendre à la cellule suivante.")

In [ ]:
import json, csv, os, zipfile, urllib.request
import numpy as np
import transformers, torch

assert transformers.__version__.startswith("4"), "il faut redémarrer la session"
print(transformers.__version__, torch.cuda.get_device_name(0))

## Les données

In [ ]:
if not os.path.exists("scifact"):
    urllib.request.urlretrieve(
        "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scifact.zip",
        "scifact.zip")
    zipfile.ZipFile("scifact.zip").extractall(".")

documents = [json.loads(l) for l in open("scifact/corpus.jsonl", encoding="utf-8")]
questions = {json.loads(l)["_id"]: json.loads(l)["text"]
             for l in open("scifact/queries.jsonl", encoding="utf-8")}

pertinence = {}
with open("scifact/qrels/test.tsv") as f:
    lignes = csv.reader(f, delimiter="\t")
    next(lignes)
    for question_id, doc_id, note in lignes:
        pertinence.setdefault(question_id, {})[doc_id] = int(note)

identifiants = [d["_id"] for d in documents]
textes = [f"{d['title']} {d['text']}".strip() for d in documents]

print(len(documents), "documents,", len(pertinence), "questions de test")

## Les métriques

nDCG@10 est la métrique du banc. Avec une pertinence binaire, le gain d'un
document pertinent au rang i vaut 1 divisé par le log du rang, et on divise par
ce qu'aurait donné le classement idéal.

In [ ]:
def ndcg(classement, pertinents, k=10):
    gains = np.array([pertinents.get(d, 0) for d in classement[:k]], dtype=float)
    obtenu = (gains / np.log2(np.arange(2, len(gains) + 2))).sum()
    ideal = np.array(sorted(pertinents.values(), reverse=True)[:k], dtype=float)
    if ideal.size == 0:
        return 0.0
    parfait = (ideal / np.log2(np.arange(2, ideal.size + 2))).sum()
    return obtenu / parfait if parfait else 0.0


def rappel(classement, pertinents, k=100):
    total = sum(1 for v in pertinents.values() if v > 0)
    if not total:
        return 0.0
    return sum(1 for d in classement[:k] if pertinents.get(d, 0) > 0) / total


scores = {}

def mesurer(classements, nom, attendu=None):
    n = np.mean([ndcg(classements[q], pertinence[q]) for q in pertinence])
    r = np.mean([rappel(classements[q], pertinence[q]) for q in pertinence])
    scores[nom] = n
    ligne = f"{nom:<32} nDCG@10 {n:.4f}   Recall@100 {r:.4f}"
    if attendu is not None:
        ecart = n - attendu
        ligne += f"   attendu {attendu:.4f}, écart {ecart:+.4f}"
        if abs(ecart) > 0.02:
            ligne += "  <- à comprendre avant de continuer"
    print(ligne)
    return n


def meilleurs(notes, k=100):
    haut = np.argpartition(-notes, k)[:k]
    return [identifiants[i] for i in haut[np.argsort(-notes[haut])]]

## BM25

Réécrit en NumPy. Les réglages sont ceux d'Anserini, k1 à 0,9 et b à 0,4, pas
ceux qu'on trouve habituellement dans les manuels. La racinisation est du Porter
et la liste de mots vides celle de Lucene, parce que c'est ce qu'utilise
Elasticsearch dans le papier BEIR.

In [ ]:
import re
from collections import Counter
from nltk.stem.porter import PorterStemmer

mots_vides = set('''a an and are as at be but by for if in into is it no not of on
or such that the their then there these they this to was will with'''.split())

racine = PorterStemmer()
deja_vu = {}

def decouper(texte):
    mots = []
    for mot in re.findall(r"[a-z0-9]+", texte.lower()):
        if mot in mots_vides:
            continue
        if mot not in deja_vu:
            deja_vu[mot] = racine.stem(mot)
        mots.append(deja_vu[mot])
    return mots


class BM25:
    def __init__(self, textes, k1=0.9, b=0.4):
        self.k1 = k1
        self.b = b
        decoupes = [decouper(t) for t in textes]
        self.nb_documents = len(decoupes)
        self.longueurs = np.array([len(d) for d in decoupes], dtype=np.float32)
        self.longueur_moyenne = self.longueurs.mean()

        occurrences = {}
        for i, mots in enumerate(decoupes):
            for mot, frequence in Counter(mots).items():
                occurrences.setdefault(mot, []).append((i, frequence))

        self.index = {}
        for mot, liste in occurrences.items():
            ou = np.array([x[0] for x in liste], dtype=np.int32)
            combien = np.array([x[1] for x in liste], dtype=np.float32)
            df = len(liste)
            rarete = np.log(1 + (self.nb_documents - df + 0.5) / (df + 0.5))
            self.index[mot] = (ou, combien, rarete)

    def noter(self, question):
        notes = np.zeros(self.nb_documents, dtype=np.float32)
        for mot in decouper(question):
            if mot not in self.index:
                continue
            ou, combien, rarete = self.index[mot]
            longueur = 1 - self.b + self.b * self.longueurs[ou] / self.longueur_moyenne
            notes[ou] += rarete * combien * (self.k1 + 1) / (combien + self.k1 * longueur)
        return notes


bm25 = BM25(textes)
notes_bm25 = {q: bm25.noter(questions[q]) for q in pertinence}
classement_bm25 = {q: meilleurs(notes_bm25[q]) for q in pertinence}
mesurer(classement_bm25, "BM25", attendu=0.665)

## Le plafond du reclassement

Si un reclasseur parfait remontait en tête tous les documents pertinents déjà
présents dans les cent premiers, jusqu'où irait-on ? Ça dit combien il reste à
gagner avant d'investir dans un modèle lourd.

In [ ]:
parfait = {}
for q in pertinence:
    liste = classement_bm25[q]
    bons = [d for d in liste if pertinence[q].get(d, 0) > 0]
    parfait[q] = bons + [d for d in liste if d not in set(bons)]
mesurer(parfait, "reclassement parfait du top-100")

## Le modèle dense

GTE-large-en-v1.5 est le meilleur des modèles publiés sur SciFact. Je le
reproduis avant de m'en servir. Un essai sur CPU d'abord : si le code maison est
cassé, il échoue là sans bloquer le GPU pour le reste de la session.

In [ ]:
from sentence_transformers import SentenceTransformer

nom_dense = "Alibaba-NLP/gte-large-en-v1.5"

essai = SentenceTransformer(nom_dense, trust_remote_code=True, device="cpu")
vecteur = essai.encode(["test"], convert_to_numpy=True)
assert vecteur.shape[1] == 1024
del essai
print("le modèle se charge, on passe sur GPU")

dense = SentenceTransformer(nom_dense, trust_remote_code=True, device="cuda")
dense.max_seq_length = 512

vecteurs_documents = dense.encode(textes, batch_size=32, normalize_embeddings=True,
                                  show_progress_bar=True, convert_to_numpy=True)
ordre_questions = list(pertinence)
vecteurs_questions = dense.encode([questions[q] for q in ordre_questions], batch_size=32,
                                  normalize_embeddings=True, convert_to_numpy=True)

notes_dense = {q: vecteurs_documents @ v for q, v in zip(ordre_questions, vecteurs_questions)}
classement_dense = {q: meilleurs(notes_dense[q]) for q in pertinence}
mesurer(classement_dense, "GTE-large-en-v1.5", attendu=0.8243)

## Fusionner les deux

BM25 retrouve le terme technique exact, le modèle dense retrouve la
reformulation. On fusionne par rang plutôt que par score, parce qu'un score BM25
n'a pas de borne et qu'un cosinus vit entre -1 et 1. La constante amortit le
poids des premiers rangs, je balaie quelques valeurs.

In [ ]:
def rangs(notes, profondeur=200):
    haut = np.argpartition(-notes, profondeur)[:profondeur]
    return {int(i): r for r, i in enumerate(haut[np.argsort(-notes[haut])])}


def fusionner(listes_de_rangs, poids, amortissement=60, k=100):
    total = {}
    for rangs_un, poids_un in zip(listes_de_rangs, poids):
        for document, rang in rangs_un.items():
            total[document] = total.get(document, 0.0) + poids_un / (amortissement + rang + 1)
    ordonne = sorted(total.items(), key=lambda x: -x[1])[:k]
    return [identifiants[i] for i, _ in ordonne]


rangs_bm25 = {q: rangs(notes_bm25[q]) for q in pertinence}
rangs_dense = {q: rangs(notes_dense[q]) for q in pertinence}

meilleur_reglage = None
meilleur_score = 0.0
for amortissement in (10, 20, 60):
    for poids_bm25 in (0.2, 0.3, 0.5, 0.7, 1.0):
        essai = {q: fusionner([rangs_bm25[q], rangs_dense[q]], [poids_bm25, 1.0], amortissement)
                 for q in pertinence}
        valeur = np.mean([ndcg(essai[q], pertinence[q]) for q in pertinence])
        print(f"amortissement {amortissement:<4} poids BM25 {poids_bm25:<5} {valeur:.4f}")
        if valeur > meilleur_score:
            meilleur_score = valeur
            meilleur_reglage = (amortissement, poids_bm25)

amortissement, poids_bm25 = meilleur_reglage
hybride = {q: fusionner([rangs_bm25[q], rangs_dense[q]], [poids_bm25, 1.0], amortissement)
           for q in pertinence}
mesurer(hybride, "BM25 + GTE")

## Le reclassement

Un cross-encodeur lit la question et le document ensemble au lieu de comparer
deux vecteurs calculés séparément. C'est plus précis en principe, et beaucoup
plus lent. Je mesure deux façons de s'en servir : remplacer l'ordre existant, ou
le fusionner avec.

In [ ]:
import time
from sentence_transformers import CrossEncoder

reclasseur = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512, device="cuda")
reclasseur.model.half()

paires = []
bornes = []
for q in pertinence:
    candidats = hybride[q][:100]
    bornes.append((q, len(paires), len(paires) + len(candidats), candidats))
    paires.extend((questions[q], textes[identifiants.index(d)]) for d in candidats)

depart = time.time()
notes_reclasseur = reclasseur.predict(paires, batch_size=128, show_progress_bar=True)
print(f"{1000 * (time.time() - depart) / len(pertinence):.0f} ms par question")

remplace = {q: [d for _, d in sorted(zip(notes_reclasseur[a:b], candidats), key=lambda x: -x[0])]
            for q, a, b, candidats in bornes}
mesurer(remplace, "reclassement, en remplacement")

position = {d: i for i, d in enumerate(identifiants)}

def rangs_de_liste(liste):
    return {position[d]: r for r, d in enumerate(liste)}

meilleur_score_ce = 0.0
meilleur_poids_ce = None
for amortissement_ce in (10, 20, 60):
    for poids_ce in (0.5, 1.0, 1.5, 2.0, 3.0):
        essai = {q: fusionner([rangs_de_liste(hybride[q]), rangs_de_liste(remplace[q])],
                              [1.0, poids_ce], amortissement_ce)
                 for q in pertinence}
        valeur = np.mean([ndcg(essai[q], pertinence[q]) for q in pertinence])
        if valeur > meilleur_score_ce:
            meilleur_score_ce = valeur
            meilleur_poids_ce = (amortissement_ce, poids_ce)

amortissement_ce, poids_ce = meilleur_poids_ce
fusionne = {q: fusionner([rangs_de_liste(hybride[q]), rangs_de_liste(remplace[q])],
                         [1.0, poids_ce], amortissement_ce)
            for q in pertinence}
mesurer(fusionne, "reclassement, en fusion")

## Est-ce que les écarts sont réels

Les systèmes sont évalués sur les mêmes questions, donc on peut apparier et
soustraire la variance entre questions, qui est énorme. Je regarde aussi sur
combien de questions chaque système gagne, ce qui est souvent plus parlant
qu'une valeur de p.

In [ ]:
def par_question(classements):
    return np.array([ndcg(classements[q], pertinence[q]) for q in pertinence])


def comparer(a, b, nom_a, nom_b, tirages=10000):
    ecarts = par_question(a) - par_question(b)
    tirage = np.random.default_rng(0).choice(len(ecarts), (tirages, len(ecarts)))
    moyennes = ecarts[tirage].mean(axis=1)
    p = 2 * min((moyennes <= 0).mean(), (moyennes >= 0).mean())
    bas, haut = np.percentile(moyennes, [2.5, 97.5])
    print(f"{nom_a} contre {nom_b}")
    print(f"  écart {ecarts.mean():+.4f}, intervalle [{bas:+.4f}, {haut:+.4f}], p = {min(p, 1):.3f}")
    print(f"  {nom_a} gagne sur {(par_question(a) > par_question(b)).sum()} questions, "
          f"{nom_b} sur {(par_question(a) < par_question(b)).sum()}")


comparer(hybride, classement_dense, "hybride", "GTE seul")
comparer(fusionne, hybride, "avec reclassement", "hybride")

## Récapitulatif

In [ ]:
publies = [("BM25", 0.665), ("BM25 + cross-encodeur", 0.688), ("E5-large-v2", 0.7224),
           ("Snowflake arctic-embed-l", 0.7382), ("BGE-large-en-v1.5", 0.7461),
           ("GTE-large-en-v1.5", 0.8243)]

print(f"{'publié':<34}{'nDCG@10':>10}")
for nom, valeur in publies:
    print(f"{nom:<34}{valeur:>10.4f}")
print()
print(f"{'mesuré':<34}{'nDCG@10':>10}")
for nom, valeur in scores.items():
    print(f"{nom:<34}{valeur:>10.4f}")

json.dump({"publies": dict(publies), "mesures": {k: float(v) for k, v in scores.items()}},
          open("resultats_recherche.json", "w"), indent=2)